# L3b: Stacks and Queues

An array lets you read or write any element at any time. A stack and a queue give up that freedom: items enter and leave only at the ends, and the end you use decides the order items come back. Today we build both on ordinary Julia arrays, put each behind an interface that is the only supported way to use it, and meet the linked list, which stores order by links instead of by position and leads to trees later in the course.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Describe the behavior of stacks, queues and linked lists:__ Describe which item comes out first from a stack and which comes out first from a queue, and choose between them from the order a problem needs. Explain how an array keeps order by position while a linked list keeps order by having each item point to the next, and choose between those from the operations you will do most.
> * __Add and remove items for stacks, queues and linked lists:__ Add and remove items by calling those functions instead of touching the array inside the type. Explain why going through them is what keeps the order correct, and why the field they wrap is private by convention rather than by enforcement.
> * __Trace algorithms that use stacks and queues:__ Update a stack or queue after each operation and predict which item will be removed next. Use that trace to explain why nested calls, balanced delimiters, and undo use a stack; why commands waiting to execute use a queue; and how one workflow can use both.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
# Setup -
include(joinpath(@__DIR__, "Include.jl")); # activate the pinned course environment and load the L3b dependencies

This lecture needs nothing beyond the course environment: the [Test standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies the checks we run along the way, the course package supplies the `MyStack` and `MyQueue` types and [the `isbalanced(...)` function](../../../code/src/StacksQueues.jl) we examine, and the `MutableLinkedList` type in the linked-list section comes from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl), which the course environment already carries.
___

## Stacks: last in, first out
Stacks are a type of data structure that follows the last-in, first-out (LIFO) principle. For example, the undo feature of an editor reverses the latest edit, not the oldest one, and the back button of a browser returns to the page you just left. Both of these are stacks.

> __The rule a stack follows:__
>
> A stack accepts new items and releases items at the same end, called the top. The last item pushed is the first item popped, so a stack gives items back in reverse order. This order is called __last-in-first-out__, or __LIFO__. Think of a stack of plates: you can only add or remove the top plate, and the last plate you put on is the first one you take off.

The figure shows pushing an item onto a stack: `push!(s, 16)` places 16 on top of the stack holding 8, 4, 2, and the next `pop!(s)` removes and returns that same 16, the newest item.

<div>
    <center>
        <img src="figs/Fig-Stack.svg" width="560" alt="A stack holding 8, 4, 2 shown before and after push!(s, 16) places 16 on top, and after pop!(s) removes and returns that same 16"/>
    </center>
</div>

A plain Julia vector can already do both of these operations: [the `push!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.push!) adds an item at the back, and [the `pop!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pop!) removes and returns the item at the back. If we treat the back of the vector as the top of the stack, the pop operation will allow us to get items back in undo order. 

Let's record three edits and then undo two of them:

In [12]:
let
    # Initialize -
    edit_history = Vector{String}() # the back of this vector acts as the top of the undo stack

    # Populate - record edits from oldest to newest
    push!(edit_history, "typed the heading")
    push!(edit_history, "added the figure")
    push!(edit_history, "fixed the caption")

    # Undo -
    (first_undo = pop!(edit_history), second_undo = pop!(edit_history), still_applied = edit_history) # remove the two newest edits first
end

(first_undo = "fixed the caption", second_undo = "added the figure", still_applied = ["typed the heading"])

The two undo steps came back in reverse order of entry: the caption fix first, then the figure, with the heading still applied. That is LIFO, and it is what an undo feature needs. There is a catch, though. Nothing about a plain vector enforces the rule: any caller can remove an element from the front, and then the LIFO guarantee is gone.

Let's fix this by creating a `MyStack` type, defined in [the `StacksQueues.jl` file](../../../code/src/StacksQueues.jl). It is a [composite type](https://docs.julialang.org/en/v1/manual/types/#Composite-Types) holding the vector in a private `items` field, and its public functions are the only supported way to use it. Julia does not lock fields away, so this is a convention rather than a strict wall, but code that uses only those functions cannot break the LIFO principle.

Let's use `MyStack` for a job from Week 2: walking the characters of a string. The string is the molecular formula of glucose, and the `formula_stack::MyStack{Char}` variable holds each character of `C6H12O6` in arrival order:

In [3]:
formula_stack = let
    # Initialize -
    formula_stack = MyStack{Char}() # stack restricted to individual characters

    # Populate - visit the formula from left to right
    for character in "C6H12O6"
        push!(formula_stack, character) # each arriving character becomes the new top item
    end

    # Return -
    formula_stack # populated stack returned by the let block
end

MyStack{Char}(['C', '6', 'H', '1', '2', 'O', '6'])

Popping until the stack is empty must now return the characters in reverse arrival order. The `reversed_formula::String` variable collects them, and the loop leaves `formula_stack` empty, so re-run the cell above before running this one again:

In [ ]:
reversed_formula = let
    # Initialize -
    characters = Vector{Char}() # characters collected in stack-removal order
    should_loop_stop = false; # continue until the final stack item has been removed

    # Drain -
    while should_loop_stop == false # drains formula_stack; reconstruct it before rerunning this cell
        push!(characters, pop!(formula_stack)) # append the current LIFO top to the result

        if isempty(formula_stack)
            should_loop_stop = true; # stop after the last valid pop
        end
    end

    # Convert -
    String(characters) # join the characters in LIFO removal order into one string
end

"6O21H6C"

The formula comes back as `6O21H6C`: the final `6` was pushed last, so it is popped first. Reversal is what LIFO means, and it is why a stack is the right choice when things must be undone in the reverse order they were done.


__Call stack__: Another (hidden) example of a stack is the call stack. You have been using a stack all semester without seeing it. The function calls a program has started but not yet finished are tracked on a stack: entering a call pushes a __stack frame__ holding the local variables and the point to return to, and returning from a function pops that frame off the stack. Calls therefore unwind in reverse order, and the most recently entered function is always the first to finish.

To watch this happen, let's nest three functions and print a line on the way into and out of each one:

In [ ]:
let
    # Define - each function prints when its call frame is entered and exited
    function inner()
        println("        inner  entered last")
        println("        inner  finished first")
    end

    function middle()
        println("    middle entered second")
        inner() # middle remains active while the inner call runs
        println("    middle finished second")
    end

    function outer()
        println("outer  entered first")
        middle() # outer remains active while the middle call runs
        println("outer  finished last")
    end

    # Run -
    outer() # build the nested call chain outer -> middle -> inner
end

outer  entered first
    middle entered second
        inner  entered last
        inner  finished first
    middle finished second
outer  finished last


The entry lines print in call order and the finish lines print in reverse. This is the same push-pop pattern as the undo demo. What the printout shows is that order, but not a count of the frames actually built, i.e., how many calls we stacked up. 

There is a limit on how many frames fit, and a chain of calls that never returns keeps pushing frames until the runtime throws a [`StackOverflowError` exception](https://docs.julialang.org/en/v1/base/base/#Core.StackOverflowError), an error named for this exact stack.

___

## Queues: first in, first out
Other work has to be done in arrival order. A shared printer takes jobs in the order they were submitted, and a simulation processes events in the order they occur. Doing either most-recent-first would be wrong: latecomers would jump the line.

> __The rule a queue follows:__
>
> A queue accepts new items at the back and releases items from the front. The first item in is the first item out, so a queue keeps arrival order. This order is called __first-in-first-out__, or __FIFO__. Queues are like checkout lines: the first person in line is the first to be served, and new arrivals go to the back of the line.

Pushing `16` onto the queue adds it at the back of the line behind `8`, while the next removal serves the `2` at the front, exactly as a checkout line would.

<div>
    <center>
        <img src="figs/Fig-Queue.svg" width="580" alt="A queue holding 2, 4, 8 shown before and after push!(q, 16) joins the back of the line, and after popfirst!(q) serves and returns the 2 from the front"/>
    </center>
</div>

On a plain vector, [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!) removes and returns the first element. The `MyQueue` type in [the `StacksQueues.jl` file](../../../code/src/StacksQueues.jl) wraps that rule the same way `MyStack` wrapped LIFO: internal vector, public functions, no other supported access path.

Let's consider an example. The `formula_reading::String` variable pushes the same formula through a `MyQueue{Char}` and empties it from the front:

In [6]:
formula_reading = let
    # Initialize -
    character_queue = MyQueue{Char}() # FIFO queue restricted to individual characters

    # Populate - visit the formula from left to right
    for character in "C6H12O6"
        push!(character_queue, character) # each arriving character joins the back
    end

    # Drain -
    characters = Vector{Char}() # characters collected in queue-removal order
    while !isempty(character_queue)
        push!(characters, popfirst!(character_queue)) # remove the oldest queued character first
    end

    # Convert -
    String(characters) # join the characters in FIFO removal order into one string
end

"C6H12O6"

The queue returns `C6H12O6` in arrival order, while the stack returned `6O21H6C` in reverse arrival order. This is the difference between FIFO and LIFO applied to the same input.

> __Cost of operating at each end of a Julia vector:__
>
> Let $n$ be the number of items currently stored. Big $\Theta$ notation describes exact cost class of a computation. 
> * __Big Theta__: A __constant time__ operation, written $\Theta(1)$ and read “big theta of one,” means an operation does about the same amount of work no matter how long the vector is. A __linear time__ operation, written $\Theta(n)$ and read “big theta of $n$,” means the work grows in direct proportion to the vector length: if $n$ doubles, the work roughly doubles.
>
> Julia 1.12 keeps track of where the used portion of a vector begins, so [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!) advances that starting location instead of shifting the remaining $n-1$ items. It therefore takes constant time, $\Theta(1)$, just like [the `pop!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pop!) at the back.
>
> On the other hand, insertion is different because the vector may need more storage. Both [the `push!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.push!) at the back and [the `pushfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pushfirst!) at the front take constant time, $\Theta(1)$, when unused capacity is available at the requested end. However, if Julia needs to allocate or recenter the storage, it copies the $n$ existing items, so that one insertion takes linear time, $\Theta(n)$. Julia reserves extra capacity when it grows the vector, so the cost averaged over a long sequence of insertions is constant; this average is called __amortized constant time__, or amortized $\Theta(1)$.

Our vector-backed queue uses [the `push!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.push!) at the back and [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!) at the front. Dequeue takes constant time, $\Theta(1)$. Enqueue takes amortized constant time, but a single enqueue can take linear time, $\Theta(n)$, when the vector grows or recenters. If a program needs operations at both ends, the `Deque` type from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl) makes that access pattern explicit. The linked-list section below presents another storage layout whose front insertions do not copy the existing items.

___

## Application: checking balanced delimiters
Every Julia expression you have typed this semester follows a rule you never checked by hand: parentheses, brackets, and braces must close in last-opened-first-closed order. That is the stack rule, and a stack is how a parser checks delimiters.

Let's write the check as pseudocode:

__Initialization:__ Given a text $\mathbf{t}$, create an empty stack $\texttt{S}$ to hold the delimiters that are currently open.

For each character $c\in\mathbf{t}$ __do__:
1. If $c$ is an opener, one of `(`, `[`, or `{`, then push it onto the stack: $\texttt{S}\gets\texttt{push}(\texttt{S},c)$.
2. __Else if__ $c$ is a closer, one of `)`, `]`, or `}`, __then__:
    * If $\texttt{S}$ is empty, __return__ $\texttt{false}$, because a delimiter closed a statement that was never opened.
    * Otherwise pop the top of the stack, $o\gets\texttt{pop}(\texttt{S})$. If $o$ is not the opener that matches $c$, __return__ $\texttt{false}$, because this closer does not match the most recently opened delimiter.
3. __Else__ $c$ is not a delimiter, so ignore it and move on.

__Termination:__ __Return__ $\texttt{true}$ if $\texttt{S}$ is empty, and $\texttt{false}$ otherwise. Anything still on the stack at the end of the text was opened and never closed.

The `isbalanced(...)` function in [the `StacksQueues.jl` file](../../../code/src/StacksQueues.jl) implements this idea, and it is another example of the public-function, private-detail split. The function callers use is `isbalanced(...)`, while the `_OPENER_FOR_CLOSER` table and [the `_isopener(...)` helper function](../../../code/src/StacksQueues.jl) are private. The leading underscore is how Julia programmers mark internals a caller should not use.

One caveat: the checker reads raw characters, so a lone delimiter inside a quoted string counts like any other. A real parser strips string literals and comments before a check like this runs.

Let's test `isbalanced(...)` with [the `@testset` macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset), including inputs that should be rejected:

In [7]:
# Check - accept balanced inputs and reject three distinct failure modes
@testset "isbalanced enforces stack discipline" begin
    # Accept -
    @test isbalanced("f(x[2]) + {a: (b)}") # each closer matches the most recent opener
    @test isbalanced("no delimiters at all") # no pending delimiters remain

    # Reject -
    @test !isbalanced("f(x[2)]") # ')' cannot close the most recently opened '['
    @test !isbalanced(")(") # ')' appears before any matching opener
    @test !isbalanced("open( forever") # '(' remains on the stack at the end
end;

Test Summary:                        | Pass  Total  Time
isbalanced enforces stack discipline |    5      5  0.3s


All five results match what you get by reading the strings, and the two things doing the work are what this lecture is about: a stack to remember what is open, and an interface that keeps the caller out of the details.
___

## Using a stack and a queue together: a replay buffer
Suppose we have a survey robot exploring a planetary surface: mission control transmits movement commands, which the robot must execute these commands in the order sent. A `back` command must undo the most recent move. The execution order is a queue's job, and the undo functionality can be implemented with a stack. 

> __The design:__
>
> Commands wait in a `MyQueue{Char}` and are executed first-in-first-out, exactly as transmitted. Every executed move is also pushed onto a `MyStack{Char}` of history. A `back` command pops that history and executes the inverse of whatever comes off, so undo always applies to the most recent move, no matter when the command sequence was written.

The `replay_trace::NamedTuple` variable runs a small mission: north, north, east, east, west, then two `back` commands and a pause. Watch what each `back` undoes:

In [8]:
replay_trace = let
    # Initialize pending work -
    command_tape = MyQueue{Char}() # FIFO commands awaiting execution
    for command in "nneewbbp" # north, north, east, east, west, back, back, pause
        push!(command_tape, command) # preserve the transmission order
    end

    # Define motion rules - map each command to a Cartesian step and each move to its inverse
    displacement = Dict('n' => (0, 1), 's' => (0, -1), 'e' => (1, 0), 'w' => (-1, 0), 'p' => (0, 0))
    inverse_move = Dict('n' => 's', 's' => 'n', 'e' => 'w', 'w' => 'e')

    # Initialize execution state -
    undo_history = MyStack{Char}() # LIFO history of completed movement commands
    x, y = 0, 0 # current planar position
    mission_log = Vector{String}() # one trace entry per processed command

    # Process -
    while !isempty(command_tape)
        command = popfirst!(command_tape) # execute the oldest waiting command
        action = command # movement and pause commands execute directly
        if command == 'b'
            action = isempty(undo_history) ? 'p' : inverse_move[pop!(undo_history)] # undo the newest move, or pause when history is empty
        elseif haskey(inverse_move, command)
            push!(undo_history, command) # record only movement commands for later undo
        end

        # Update state -
        (dx, dy) = displacement[action] # Cartesian increment for the resolved action
        x += dx
        y += dy
        push!(mission_log, "command $(command) -> action $(action), position ($(x), $(y))") # record the command, action, and resulting position
    end

    # Return -
    (final_position = (x, y), mission_log = mission_log) # final state and complete execution trace
end

(final_position = (1, 2), mission_log = ["command n -> action n, position (0, 1)", "command n -> action n, position (0, 2)", "command e -> action e, position (1, 2)", "command e -> action e, position (2, 2)", "command w -> action w, position (1, 2)", "command b -> action e, position (2, 2)", "command b -> action w, position (1, 2)", "command p -> action p, position (1, 2)"])

The log shows the first `back` undoing the westward step and the second undoing an eastward one: most recent first, and that is the stack. The commands themselves ran in exactly the order they were sent, and that is the queue.

Using a queue for what to do and a stack for what was done is common in systems that run commands in order and undo them in reverse.
___

## Linked lists

Our `MyStack` and `MyQueue` types kept their items in one contiguous array, and the queue's cost note showed the drawback: a front insertion can require copying all $n$ existing items. A __linked list__ can perform that insertion without moving the items already stored.

> __How each layout stores order:__
>
> An array stores order by position. The fifth element sits four slots past the first, so you can jump straight to it. A linked list stores order by links, so to reach the fifth element you follow four links from the head. An array is faster when you want to read an element by its position. A linked list is faster when you want to add or remove an item at a node you already have, since nothing else moves.


Each item lives in its own small __node__, and a node holds a value and a reference to the node that follows it. The list itself remembers only the __head__, the first node. The last node references nothing, which is how you know you have reached the end.

<div>
    <center>
        <img src="figs/Fig-LinkedList.svg" width="700" alt="A linked list holding 2, 4, 8: the head references the first node, each node holds a value and a reference to the node that follows it, the last node references nothing, and a second row shows 16 spliced in after 4 with two relinks while nothing else moves"/>
    </center>
</div>


The splice in the figure is the operation arrays are slowest at, and a linked list does it by updating two references. The three figures now line up: the stack pushed 16 on top, the queue added it at the back, and the linked list can put it anywhere.

We do not have to build one by hand. The `MutableLinkedList` type from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl) is a ready-made linked list, and it is __doubly linked__: each node also references the node before it, so a splice updates links in both directions rather than the one the schematic draws. The extra reference uses more memory and lets you walk the list backwards.

Let's load the glucose formula into a linked list. The `formula_list::MutableLinkedList{Char}` variable holds one node per character, each linked after the last:

In [10]:
formula_list = let
    # Initialize -
    formula_list = MutableLinkedList{Char}() # doubly linked nodes holding individual characters

    # Populate - visit the formula from left to right
    for character in "C6H12O6"
        push!(formula_list, character) # link each new node after the current tail
    end

    # Return -
    formula_list # populated list returned by the let block
end

MutableLinkedList{Char}(C, 6, H, 1, 2, O, 6)

Iterating the list follows the references from the head, so the characters come back in arrival order, no array required. Adding at the front costs the same here every time, where a vector sometimes has to move every element it already holds to make room.

The `front_edit::NamedTuple` variable pushes a stray character on with [the `pushfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pushfirst!) and removes it with [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!). Neither one moves an existing node, however long the list has grown, and then we read the whole list back:

In [11]:
front_edit = let
    # Edit front -
    pushfirst!(formula_list, '?') # link a new head without moving the existing nodes
    removed = popfirst!(formula_list) # unlink that head without moving the remaining nodes

    # Return -
    (removed = removed, reading = String(collect(formula_list))) # traverse the restored list from head to tail
end

(removed = '?', reading = "C6H12O6")

The reading is `C6H12O6`, intact and in order, and neither front operation moved another node.

This idea is where we go next. A linked-list node references exactly one successor. Let a node reference several, start from one root, and let every other node be referenced exactly once, and the chain becomes a __tree__. Tomorrow's lecture draws a recursive computation as a tree of calls, and the active path through that tree lives on the call stack we just watched.

Next week, graph search walks structures built from this node-and-reference idea, holding its __frontier__, the places it still has to visit, in a container you choose. Make it a queue and the search spreads outward level by level, make it a stack and it dives deep before backing up. Same algorithm, different rule, different traversal.
___

## Summary
A stack or queue specifies an access order; an array or linked list specifies a storage layout and its costs. Those are separate design decisions.

> __Key Takeaways:__
>
> * __Choose the access rule from the next item the problem needs:__ A stack adds and removes at the same end, so the newest item leaves first (LIFO). A queue adds at the back and removes at the front, so the oldest item leaves first (FIFO).
> * __Trace the container to explain an algorithm:__ Nested calls, delimiter matching, and undo all need the most recent unfinished item, so they use a stack. Waiting commands must execute in arrival order, so they use a queue. The replay buffer combines a FIFO command queue with a LIFO undo stack.
> * __Use the public operations to preserve the rule:__ `MyStack` and `MyQueue` provide operations consistent with their access order. Their `items` fields remain reachable because Julia privacy is a convention, but client code should not modify those fields directly.
> * __Choose the storage layout from the operations and costs:__ A vector provides constant-time indexed access, written $\Theta(1)$, but an insertion can occasionally take linear time, $\Theta(n)$, because it copies all $n$ items when the storage changes. A linked list takes linear time to reach the $n$th item, but a front insertion or removal changes only a fixed number of links and therefore takes constant time.

Pick the rule from the order the problem needs, and pick the storage from the operations you will do most.
___